In [6]:
import os
import json
import ast
from radon.complexity import cc_visit
import subprocess
import sys
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import re

# --- Configuration ---
AI_MODEL_PATH = "./trained-generative-model"
AI_PREFIX = "suggest bug: "
MAX_INPUT_LENGTH = 512
MAX_OUTPUT_LENGTH = 128

# --- Enhanced AI Suggestion Function ---
def get_ai_suggestion(code_block, tokenizer, model, function_name=""):
    """
    Gets suggestions from trained model with better preprocessing and post-processing.
    """
    try:
        # Normalize whitespace
        normalized_code = code_block.replace('\t', '    ')
        
        # Create prompt with context
        prompt = f"{AI_PREFIX}{normalized_code}"
        
        inputs = tokenizer(
            prompt, 
            return_tensors="pt", 
            max_length=MAX_INPUT_LENGTH, 
            truncation=True,
            padding=True
        )

        with torch.no_grad():
            output_ids = model.generate(
                inputs.input_ids,
                max_length=MAX_OUTPUT_LENGTH,
                num_beams=5,
                early_stopping=True,
                no_repeat_ngram_size=2
            )
        
        suggestion = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
        
        # Post-process the suggestion
        cleaned_suggestion = clean_ai_output(suggestion, code_block)
        
        return cleaned_suggestion if cleaned_suggestion else None
        
    except Exception as e:
        print(f"  ⚠️ AI model error: {e}")
        return None


def clean_ai_output(suggestion, original_code):
    """
    Cleans up AI output to make it more readable and useful.
    """
    if not suggestion or len(suggestion) < 10:
        return None
    
    # Remove code snippets that just repeat the input
    if suggestion.strip() in original_code:
        return None
    
    # Fix common AI output issues
    suggestion = suggestion.replace("This function may raise a TypeError", 
                                   "Potential TypeError")
    suggestion = suggestion.replace("This function can lead to unexpected behavior", 
                                   "May cause unexpected behavior")
    suggestion = suggestion.replace("Consider adding a try-except block", 
                                   "Add error handling")
    
    # Remove incomplete sentences (ends with "else:" or similar)
    if re.search(r'(else|if|for|while)\s*:\s*$', suggestion):
        # Extract only the meaningful part before the code fragment
        parts = re.split(r'\s+(if|else|for|while|def|class)\s+', suggestion)
        if parts:
            suggestion = parts[0].strip()
    
    # Capitalize first letter
    if suggestion:
        suggestion = suggestion[0].upper() + suggestion[1:]
    
    # Ensure it ends with proper punctuation
    if suggestion and suggestion[-1] not in '.!?':
        suggestion += '.'
    
    return suggestion if len(suggestion) > 10 else None


# --- Enhanced Rule-Based Bug Detection ---
def analyze_code_patterns(code_content):
    """
    Advanced rule-based analysis for common bugs and security issues.
    """
    issues = []
    
    try:
        tree = ast.parse(code_content)
        
        for node in ast.walk(tree):
            # 1. Division without ZeroDivisionError handling
            if isinstance(node, ast.BinOp) and isinstance(node.op, ast.Div):
                issues.append({
                    'line': node.lineno,
                    'type': 'potential_zero_division',
                    'message': 'Division without zero check - add validation: if denominator == 0',
                    'severity': 'high',
                    'fix': 'Add check before division: if count == 0: return 0 (or raise error)'
                })
            
            # 2. Bare except clauses
            if isinstance(node, ast.ExceptHandler) and node.type is None:
                issues.append({
                    'line': node.lineno,
                    'type': 'bare_except',
                    'message': 'Bare except catches all exceptions including KeyboardInterrupt',
                    'severity': 'medium',
                    'fix': 'Use specific exception: except (IOError, ValueError) as e:'
                })
            
            # 3. range(len()) anti-pattern
            if isinstance(node, ast.Call):
                if (isinstance(node.func, ast.Name) and node.func.id == 'range' and
                    len(node.args) == 1 and isinstance(node.args[0], ast.Call)):
                    inner_call = node.args[0]
                    if isinstance(inner_call.func, ast.Name) and inner_call.func.id == 'len':
                        issues.append({
                            'line': node.lineno,
                            'type': 'range_len_pattern',
                            'message': 'Use enumerate() instead of range(len()) for cleaner code',
                            'severity': 'low',
                            'fix': 'Replace: for i, item in enumerate(items): print(item)'
                        })
            
            # 4. os.system() security risk
            if isinstance(node, ast.Call):
                if isinstance(node.func, ast.Attribute):
                    if (isinstance(node.func.value, ast.Name) and 
                        node.func.value.id == 'os' and 
                        node.func.attr == 'system'):
                        issues.append({
                            'line': node.lineno,
                            'type': 'security_os_system',
                            'message': 'os.system() is unsafe - vulnerable to command injection',
                            'severity': 'critical',
                            'fix': 'Use subprocess.run() with shell=False and list arguments'
                        })
            
            # 5. Hardcoded passwords/secrets
            if isinstance(node, ast.Assign):
                for target in node.targets:
                    if isinstance(target, ast.Name):
                        var_name = target.id.lower()
                        if any(keyword in var_name for keyword in ['password', 'secret', 'token', 'key', 'api']):
                            if isinstance(node.value, ast.Constant) and isinstance(node.value.value, str):
                                issues.append({
                                    'line': node.lineno,
                                    'type': 'hardcoded_secret',
                                    'message': f'Hardcoded credential in {target.id} - security risk',
                                    'severity': 'critical',
                                    'fix': 'Use environment variables: os.getenv("DATABASE_PASSWORD")'
                                })
            
            # 6. Unclosed file handles
            if isinstance(node, ast.Call):
                if isinstance(node.func, ast.Name) and node.func.id == 'open':
                    # Check if it's not in a 'with' statement
                    parent = getattr(node, 'parent', None)
                    if not isinstance(parent, ast.withitem):
                        issues.append({
                            'line': node.lineno,
                            'type': 'unclosed_file',
                            'message': 'File opened without context manager - resource leak',
                            'severity': 'medium',
                            'fix': 'Use: with open(filepath, "r") as f: data = f.read()'
                        })
    
    except Exception as e:
        print(f"  ⚠️ AST analysis error: {e}")
    
    return issues


# --- Extract Functions with Context ---
def extract_functions(code_content):
    """
    Extract individual functions from code for targeted analysis.
    """
    functions = []
    try:
        tree = ast.parse(code_content)
        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                func_lines = code_content.split('\n')[node.lineno - 1:node.end_lineno]
                func_code = '\n'.join(func_lines)
                
                # Calculate basic metrics
                params = len(node.args.args)
                returns = any(isinstance(n, ast.Return) for n in ast.walk(node))
                
                functions.append({
                    'name': node.name,
                    'code': func_code,
                    'lineno': node.lineno,
                    'params': params,
                    'has_return': returns
                })
    except Exception as e:
        print(f"  ⚠️ Function extraction error: {e}")
    
    return functions



file_name = "temp_code.py"

print("=" * 70)
print("🚀 CodeSage AI Code Reviewer - Professional Edition")
print("=" * 70)

# --- Load AI Model ---
print("\n--- 🧠 Loading AI Model ---")
ai_available = False
try:
    ai_tokenizer = AutoTokenizer.from_pretrained(AI_MODEL_PATH)
    ai_model = AutoModelForSeq2SeqLM.from_pretrained(AI_MODEL_PATH)
    ai_model.eval()
    print("  ✅ AI model loaded successfully")
    ai_available = True
except Exception as e:
    print(f"  ⚠️ AI model not available: {e}")
    print("  → Using rule-based analysis only")

# --- Read Code ---
try:
    with open(file_name) as f:
        code_content = f.read()
except Exception as e:
    print(f"❌ Error reading file: {e}")
    sys.exit(1)

# --- Radon Complexity Analysis ---
print("\n--- 📈 Code Complexity Analysis ---")
try:
    radon_results = cc_visit(code_content)
    
    if radon_results:
        for item in radon_results:
            complexity = item.complexity
            if complexity <= 5:
                status = "✅ Simple"
                color = ""
            elif complexity <= 10:
                status = "⚠️ Moderate"
                color = " - consider refactoring"
            else:
                status = "❌ Complex"
                color = " - NEEDS refactoring"
            print(f"  {status} '{item.name}' → Complexity: {complexity}{color}")
except Exception as e:
    print(f"  ⚠️ Radon analysis failed: {e}")

# --- Pattern-Based Bug Detection ---
print("\n--- 🔍 Static Analysis (Pattern Detection) ---")
pattern_issues = analyze_code_patterns(code_content)

if not pattern_issues:
    print("  ✅ No pattern-based issues detected")
else:
    # Sort by severity
    severity_order = {'critical': 0, 'high': 1, 'medium': 2, 'low': 3}
    pattern_issues.sort(key=lambda x: severity_order.get(x['severity'], 4))
    
    for issue in pattern_issues:
        severity_map = {
            'critical': '🔴 CRITICAL',
            'high': '🟠 HIGH',
            'medium': '🟡 MEDIUM',
            'low': '🔵 LOW'
        }
        severity_label = severity_map.get(issue['severity'], '⚪ INFO')
        
        print(f"\n  {severity_label} | Line {issue['line']}")
        print(f"    Issue: {issue['message']}")
        if 'fix' in issue:
            print(f"    Fix: {issue['fix']}")

# --- AI-Powered Analysis ---
print("\n--- 🤖 AI-Powered Deep Analysis ---")

if ai_available:
    functions = extract_functions(code_content)
    
    if not functions:
        print("  ⚠️ No functions found to analyze")
    else:
        ai_suggestions_count = 0
        
        for func in functions:
            print(f"\n  📝 Function: '{func['name']}' (Line {func['lineno']}, {func['params']} params)")
            
            suggestion = get_ai_suggestion(func['code'], ai_tokenizer, ai_model, func['name'])
            
            if suggestion:
                print(f"    💡 AI Insight: {suggestion}")
                ai_suggestions_count += 1
            else:
                print(f"    ✓ No issues detected by AI")
        
        if ai_suggestions_count == 0:
            print("\n  ℹ️ AI model didn't find additional issues beyond pattern analysis")
else:
    print("  ⚠️ AI analysis skipped (model not available)")

# --- Summary Report ---
print("\n" + "=" * 70)
print("📊 Analysis Summary Report")
print("=" * 70)

critical_count = sum(1 for i in pattern_issues if i['severity'] == 'critical')
high_count = sum(1 for i in pattern_issues if i['severity'] == 'high')
medium_count = sum(1 for i in pattern_issues if i['severity'] == 'medium')
low_count = sum(1 for i in pattern_issues if i['severity'] == 'low')

print(f"  📌 Total Issues Found: {len(pattern_issues)}")
print(f"     • 🔴 Critical: {critical_count}")
print(f"     • 🟠 High: {high_count}")
print(f"     • 🟡 Medium: {medium_count}")
print(f"     • 🔵 Low: {low_count}")
print(f"\n  🔧 Functions Analyzed: {len(extract_functions(code_content))}")
print(f"  🤖 AI Model Status: {'✅ Active' if ai_available else '⚠️ Inactive'}")

if critical_count > 0:
    print(f"\n  ⚠️ ACTION REQUIRED: {critical_count} critical security issue(s) found!")




🚀 CodeSage AI Code Reviewer - Professional Edition

--- 🧠 Loading AI Model ---
  ✅ AI model loaded successfully

--- 📈 Code Complexity Analysis ---
  ✅ Simple 'add_item' → Complexity: 1

--- 🔍 Static Analysis (Pattern Detection) ---
  ✅ No pattern-based issues detected

--- 🤖 AI-Powered Deep Analysis ---

  📝 Function: 'add_item' (Line 1, 2 params)
    💡 AI Insight: _add_item_to_a_list can raise a TypeError if 'list' is None or the list is empty. Consider adding a 'None' as the default.

📊 Analysis Summary Report
  📌 Total Issues Found: 0
     • 🔴 Critical: 0
     • 🟠 High: 0
     • 🟡 Medium: 0
     • 🔵 Low: 0

  🔧 Functions Analyzed: 1
  🤖 AI Model Status: ✅ Active


In [11]:
import os
import json
import ast
from radon.complexity import cc_visit
import sys
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import re
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

# --- Configuration ---
AI_MODEL_PATH = "./trained-generative-model"
RULES_DATABASE = "learned_rules.json"
MAX_WORKERS = 4  # Parallel processing
USE_GPU = torch.cuda.is_available()
BATCH_SIZE = 8  # Process multiple functions at once
ENABLE_AI = True  # Set to False to skip AI entirely (for speed)
AI_CONFIDENCE_THRESHOLD = 0.7  # Only show high-confidence AI results

# Known patterns (instant detection)
FAST_PATTERNS = {
    r'\/\s*\w+': ("Potential division by zero", "high"),
    r'os\.system\s*\(': ("Command injection risk - use subprocess.run()", "critical"),
    r'eval\s*\(': ("Arbitrary code execution risk - avoid eval()", "critical"),
    r'exec\s*\(': ("Arbitrary code execution risk - avoid exec()", "critical"),
    r'range\s*\(\s*len\s*\(': ("Use enumerate() instead of range(len())", "low"),
    r'except\s*:': ("Bare except catches all exceptions", "medium"),
    r'pickle\.loads?\s*\(': ("Pickle deserialization security risk", "high"),
    r'password\s*=\s*[\'"]': ("Hardcoded credential detected", "critical"),
    r'open\s*\([^)]*\)(?!\s*as)': ("File not properly closed - use 'with'", "medium"),
}

# --- Fast Pattern Matcher ---
def quick_scan(code_content):
    """Ultra-fast regex-based scanning (< 10ms)."""
    issues = []
    lines = code_content.split('\n')
    
    for i, line in enumerate(lines, 1):
        for pattern, (message, severity) in FAST_PATTERNS.items():
            if re.search(pattern, line):
                issues.append({
                    'line': i,
                    'message': message,
                    'severity': severity,
                    'type': 'pattern'
                })
    
    return issues


# --- Optimized AI Detector ---
class FastAIDetector:
    """Optimized AI detector with caching and batching."""
    
    def __init__(self, model_path):
        self.model_path = model_path
        self.model = None
        self.tokenizer = None
        self.cache = {}  # Cache AI results
        self.available = self._load_model()
    
    def _load_model(self):
        """Load model once and keep in memory."""
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_path)
            self.model = AutoModelForSeq2SeqLM.from_pretrained(self.model_path)
            self.model.eval()
            
            # Move to GPU if available
            if USE_GPU:
                self.model = self.model.cuda()
                print("  🚀 Using GPU acceleration")
            
            return True
        except Exception as e:
            return False
    
    def analyze_batch(self, code_blocks):
        """Process multiple code blocks at once (much faster)."""
        if not self.available or not code_blocks:
            return [None] * len(code_blocks)
        
        results = []
        
        # Check cache first
        uncached_blocks = []
        uncached_indices = []
        
        for i, code in enumerate(code_blocks):
            cache_key = hash(code)
            if cache_key in self.cache:
                results.append(self.cache[cache_key])
            else:
                uncached_blocks.append(code)
                uncached_indices.append(i)
                results.append(None)
        
        if not uncached_blocks:
            return results
        
        try:
            # Batch tokenization (faster than one-by-one)
            prompts = [f"suggest bug: {code}" for code in uncached_blocks]
            
            inputs = self.tokenizer(
                prompts,
                return_tensors="pt",
                max_length=256,  # Reduced from 512
                truncation=True,
                padding=True
            )
            
            if USE_GPU:
                inputs = {k: v.cuda() for k, v in inputs.items()}
            
            # Batch inference (MUCH faster)
            with torch.no_grad():
                output_ids = self.model.generate(
                    inputs.input_ids,
                    max_length=64,  # Reduced from 128
                    num_beams=2,    # Reduced from 5
                    early_stopping=True,
                    do_sample=False
                )
            
            # Decode all at once
            suggestions = self.tokenizer.batch_decode(output_ids, skip_special_tokens=True)
            
            # Process and cache results
            for idx, suggestion in zip(uncached_indices, suggestions):
                cleaned = self._clean_output(suggestion)
                results[idx] = cleaned
                
                # Cache for future use
                cache_key = hash(code_blocks[idx])
                self.cache[cache_key] = cleaned
        
        except Exception as e:
            pass
        
        return results
    
    def _clean_output(self, suggestion):
        """Quick cleaning with strict quality filter."""
        if not suggestion or len(suggestion) < 15:
            return None
        
        suggestion = suggestion.strip()
        
        # Reject code snippets
        if any(x in suggestion for x in ["def ", "for ", "if ", "import ", "return "]):
            return None
        
        # Reject contradictory suggestions
        if re.search(r'instead of (\w+).*\1', suggestion, re.IGNORECASE):
            return None  # e.g., "use eval() instead of eval()"
        
        # Reject vague/repetitive suggestions
        bad_phrases = [
            "consider adding a 'echo' command",
            "use.*instead.*instead",
            "consider adding a try-except block instead",
            "as a fallback",
            "to the shell instead",
            "to the command line"
        ]
        
        for pattern in bad_phrases:
            if re.search(pattern, suggestion, re.IGNORECASE):
                return None
        
        # Must contain actionable verbs
        actionable = ['add', 'remove', 'use', 'avoid', 'replace', 'check', 
                      'validate', 'sanitize', 'fix', 'prevent', 'handle']
        
        has_action = any(verb in suggestion.lower() for verb in actionable)
        if not has_action:
            return None
        
        # Format nicely
        if suggestion[0].islower():
            suggestion = suggestion[0].upper() + suggestion[1:]
        
        if not suggestion.endswith('.'):
            suggestion += '.'
        
        # Final length check
        return suggestion if 15 < len(suggestion) < 200 else None


# --- Minimal Rule Engine ---
class LightweightRuleEngine:
    """Simplified rule engine for speed."""
    
    def __init__(self, rules_file=RULES_DATABASE):
        self.rules_file = rules_file
        self.rules = self.load_rules()
    
    def load_rules(self):
        if os.path.exists(self.rules_file):
            try:
                with open(self.rules_file, 'r') as f:
                    return json.load(f)
            except:
                pass
        return {"patterns": []}
    
    def save_rules(self):
        with open(self.rules_file, 'w') as f:
            json.dump(self.rules, f, indent=2)
    
    def add_rule(self, description, severity):
        """Simplified rule addition."""
        # Check for duplicate
        for rule in self.rules["patterns"]:
            if rule["description"] == description:
                return
        
        self.rules["patterns"].append({
            "id": f"learned_{len(self.rules['patterns']) + 1}",
            "description": description,
            "severity": severity,
            "learned_at": datetime.now().isoformat()
        })


# --- Fast AST Analysis ---
def fast_ast_scan(code_content):
    """Quick AST-based checks."""
    issues = []
    
    try:
        tree = ast.parse(code_content)
        
        for node in ast.walk(tree):
            # Division check
            if isinstance(node, ast.BinOp) and isinstance(node.op, ast.Div):
                issues.append({
                    'line': node.lineno,
                    'message': 'Division without zero check',
                    'severity': 'high'
                })
            
            # Hardcoded secrets
            if isinstance(node, ast.Assign):
                for target in node.targets:
                    if isinstance(target, ast.Name):
                        if any(kw in target.id.lower() for kw in ['password', 'secret', 'token', 'api_key']):
                            if isinstance(node.value, ast.Constant):
                                issues.append({
                                    'line': node.lineno,
                                    'message': 'Hardcoded credential - security risk',
                                    'severity': 'critical'
                                })
    except:
        pass
    
    return issues


# --- Extract Functions (Optimized) ---
def extract_functions_fast(code_content):
    """Quick function extraction."""
    functions = []
    try:
        tree = ast.parse(code_content)
        lines = code_content.split('\n')
        
        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                start = node.lineno - 1
                end = node.end_lineno if hasattr(node, 'end_lineno') else start + 10
                
                functions.append({
                    'name': node.name,
                    'code': '\n'.join(lines[start:end]),
                    'lineno': node.lineno
                })
    except:
        pass
    
    return functions


# --- Main Analysis (Optimized) ---
sample_code = """
import os

DATABASE_PASSWORD = "admin123"

def divide_numbers(a, b):
    return a / b

def run_command(cmd):
    os.system(f"echo {cmd}")

def process_items(items):
    for i in range(len(items)):
        print(items[i])

def load_file(path):
    f = open(path, 'r')
    data = f.read()
    return data

def calculate(x, y):
    return x / y

def execute_code(code_string):
    return eval(code_string)
"""

file_name = "temp.py"
with open(file_name, "w") as f:
    f.write(sample_code.strip())

print("=" * 70)
print("🚀 CodeSage - Fast AI Code Reviewer")
print("=" * 70)

start_time = time.time()

# --- Initialize (once) ---
print("\n--- 🧠 Initializing (one-time setup) ---")
init_start = time.time()

rule_engine = LightweightRuleEngine()
ai_detector = FastAIDetector(AI_MODEL_PATH)

init_time = time.time() - init_start
print(f"  ✅ Ready in {init_time:.2f}s")
print(f"  📚 {len(rule_engine.rules.get('patterns', []))} learned rules")
print(f"  {'✅' if ai_detector.available else '⚠️'} AI: {'Active' if ai_detector.available else 'Unavailable'}")

with open(file_name) as f:
    code_content = f.read()

# --- Phase 1: Fast Pattern Scan (< 50ms) ---
print("\n--- ⚡ Fast Pattern Scan ---")
scan_start = time.time()

pattern_issues = quick_scan(code_content)
ast_issues = fast_ast_scan(code_content)
all_issues = pattern_issues + ast_issues

scan_time = time.time() - scan_start
print(f"  ✅ Found {len(all_issues)} issues in {scan_time*1000:.0f}ms")

# --- Phase 2: Complexity (fast) ---
print("\n--- 📈 Complexity ---")
try:
    complex_funcs = [item for item in cc_visit(code_content) if item.complexity > 5]
    if complex_funcs:
        print(f"  ⚠️ {len(complex_funcs)} complex functions")
    else:
        print(f"  ✅ All functions simple")
except:
    pass

# --- Phase 3: Batch AI Analysis (if available and enabled) ---
if ENABLE_AI and ai_detector.available:
    print("\n--- 🤖 AI Analysis (Batched) ---")
    ai_start = time.time()
    
    functions = extract_functions_fast(code_content)
    
    if functions:
        # Process ALL functions in one batch
        codes = [f['code'] for f in functions]
        suggestions = ai_detector.analyze_batch(codes)
        
        ai_count = 0
        for func, suggestion in zip(functions, suggestions):
            if suggestion:
                print(f"  💡 {func['name']}: {suggestion}")
                all_issues.append({
                    'line': func['lineno'],
                    'message': suggestion,
                    'severity': 'medium'
                })
                rule_engine.add_rule(suggestion, 'medium')
                ai_count += 1
        
        ai_time = time.time() - ai_start
        
        if ai_count > 0:
            print(f"  ✅ Analyzed {len(functions)} functions in {ai_time:.2f}s ({ai_count} quality issues)")
        else:
            print(f"  ℹ️ Analyzed {len(functions)} functions - no additional issues beyond pattern detection")
    
    rule_engine.save_rules()
elif not ENABLE_AI:
    print("\n--- 🤖 AI Analysis ---")
    print("  ⏭️  Skipped (disabled for speed - set ENABLE_AI=True to enable)")
else:
    print("\n--- 🤖 AI Analysis ---")
    print("  ⚠️ Skipped (model unavailable)")

# --- Summary ---
total_time = time.time() - start_time

print("\n" + "=" * 70)
print("📊 Summary")
print("=" * 70)

severity_counts = {}
for issue in all_issues:
    sev = issue.get('severity', 'medium')
    severity_counts[sev] = severity_counts.get(sev, 0) + 1

print(f"  Total Issues: {len(all_issues)}")
for sev in ['critical', 'high', 'medium', 'low']:
    if sev in severity_counts:
        icon = {'critical':'🔴', 'high':'🟠', 'medium':'🟡', 'low':'🔵'}[sev]
        print(f"    {icon} {sev.capitalize()}: {severity_counts[sev]}")

print(f"\n  ⚡ Total Time: {total_time:.2f}s")
print(f"  📚 Learned Rules: {len(rule_engine.rules.get('patterns', []))}")

# --- Show critical issues ---
critical = [i for i in all_issues if i.get('severity') == 'critical']
if critical:
    print(f"\n--- 🔴 Critical Issues ---")
    for issue in critical[:5]:
        print(f"  Line {issue['line']}: {issue['message']}")

try:
    os.remove(file_name)
except:
    pass

print("\n✅ Analysis complete!\n")

🚀 CodeSage - Fast AI Code Reviewer

--- 🧠 Initializing (one-time setup) ---
  ✅ Ready in 0.54s
  📚 10 learned rules
  ✅ AI: Active

--- ⚡ Fast Pattern Scan ---
  ✅ Found 9 issues in 2ms

--- 📈 Complexity ---
  ✅ All functions simple

--- 🤖 AI Analysis (Batched) ---
  ℹ️ Analyzed 6 functions - no additional issues beyond pattern detection

📊 Summary
  Total Issues: 9
    🔴 Critical: 3
    🟠 High: 4
    🟡 Medium: 1
    🔵 Low: 1

  ⚡ Total Time: 9.83s
  📚 Learned Rules: 10

--- 🔴 Critical Issues ---
  Line 9: Command injection risk - use subprocess.run()
  Line 24: Arbitrary code execution risk - avoid eval()
  Line 3: Hardcoded credential - security risk

✅ Analysis complete!

